# Performance

Sampai saat ini, kita belum benar-benar menaruh perhatian besar pada performa kode di luar sekadar penghitungan instruksi dasar. Kuliah-kuliah ini didedikasikan untuk mendalami isu-isu tersebut dan membahas hal-hal penting apa saja yang perlu diingat.


## Example: Matrix-Matrix Multiplication

Untuk memulai diskusi kita, mari kita pertimbangkan algoritma perkalian matriks-matriks yang secara algoritmis terlihat seperti..
```
do i=1:N
    do j=1:N
        do k=1:N
            C[i, j] = C[i, j] + A[i, k] * B[k, j]
        end do
    end do
end do
```

Pertimbangkan pendekatan-pendekatan berikut untuk masalah ini:

1. Perkalian matriks melalui fungsi intrinsik GCC (Fortran)
2. Perkalian tiga loop langsung (straight forward).
3. Loop ganda yang diparalelkan menggunakan fungsi intrinsik BLAS.
4. Fungsi intrinsik BLAS.

```fortran
real function matrix_multiply_test(N,method)

    use mod_rand
    implicit none
    
    external DGEMM,DDOT
    
    double precision :: DDOT
    integer, intent(in) :: N,method
    integer :: start,finish,count_rate
    double precision, dimension(:,:), allocatable :: A,B,C
    
    ! Local
    integer :: i,j,k
    
    ! Create the random arrays
    call init_random_seed()
    allocate(A(N,N),B(N,N),C(N,N))
    call random_number(A)
    call random_number(B)
    
    ! Start the timer and start multiplying
    call system_clock(start,count_rate)
    select case(method)
        case(1) ! Default method provided as an intrinsic method
            C = matmul(A,B)
        case(2) ! Simple three loop multiplication
            !$OMP parallel do private(j,k)
            do i=1,N
                do j=1,N
                    do k=1,N
                        C(i,j) = C(i,j) + A(i,k) * B(k,j)
                    enddo
                enddo
            enddo
        case(3) ! OpenMP parallelized double loop
            !$OMP parallel do private(j)
            do i=1,N
                do j=1,N
                    C(i,j) = DDOT(N, A(i,:), 1, B(:,j), 1)
                enddo
            enddo
        case(4) ! BLAS Routine call
            ! call DGEMM(transa,transb,l,n,m,alpha,a,lda,b,ldb,beta,c,ldc)
            call DGEMM('N', 'N', N, N, N, 1.d0, A, N, B, N, 0.d0, C, N)
        case default
            print *, "***ERROR*** Invalid multiplication method!"
            matrix_multiply_test = -1
            return
    end select
    call system_clock(finish,count_rate)
    
    matrix_multiply_test = float(finish - start) / float(count_rate)
    
end function matrix_multiply_test
    
program matrix_multiply
    
    use omp_lib

    implicit none
    
    integer :: N, method, threads
    character(len=10) :: input_N, input_method, input_threads
    real :: matrix_multiply_test, time
    
    select case(iargc())
        case(0)
            N = 1000
            method = 1
            threads = 1
        case(1)
            call getarg(1,input_N)
            read(input_N,'(I10)') N
            method = 1
        case(2)
            call getarg(1,input_N)
            call getarg(2,input_method)
            read(input_N,'(I10)') N
            read(input_method,'(I10)') method
        case(3)
            call getarg(1,input_N)
            call getarg(2,input_method)
            call getarg(3,input_threads)
            read(input_N,'(I10)') N
            read(input_method,'(I10)') method
            read(input_threads,'(I10)') threads
        case default
            print *, "***ERROR*** Too many arguments!"
            stop
    end select
    
    !$ call omp_set_num_threads(threads)

    time = matrix_multiply_test(N, method)
    
    print '("Timing for ",i5,"x",i5," matrices: ",f10.5," s")',N,N,time
    
end program matrix_multiply
```

### Results

Berdasarkan perkalian matriks-matriks $1000 \times 1000$ yang dikompilasi dengan`gfortran` versi 6.3.0 menggunakan flag kompilasi`-O3 -funroll-loops -finline-functions -fdefault-real-8 -fopenmp`.


Metode                           | Tanpa Thread            | Dengan Thread
---------------------------------|-----------------------|---------------------------
Default mat_mult function        |            35.79600 s |                36.19100 s
3 loop multiplication            |            39.24700 s |                10.04000 s                   
Double loop (internal BLAS)      |             6.80500 s |                 1.76600 s                   
BLAS Routine call                |             0.00300 s |                 0.00300 s                   


## Parallelization

Paralelisasi adalah salah satu cara utama untuk meningkatkan performa pada arsitektur komputasi saat ini. Terdapat 2+1 jenis paradigma paralelisasi utama:
 - Memori Bersama (Shared memory): setiap pipeline dapat mengakses memori untuk seluruh masalah yang dikerjakan.
 - Memori Terdistribusi (Distributed memory): setiap pipeline hanya dapat mengakses sebagian memori untuk seluruh masalah tersebut.
 - Paralelisme Hibrida (Hybrid parallelism): menggunakan keduanya.

### Shared Memory

 - Konstruksi dasarnya adalah thread — setiap thread memiliki pipeline dan dalam kasus paling sederhana, setiap core menjalankan satu thread.
 - OpenMP, CUDA, OpenCL, OpenACC.
 - Node tunggal pada sebuah cluster, GPU, Xeon Phi, dll.

### Distributed Memory

 - Konstruksi dasarnya adalah process (proses) 
 - Setiap proses memiliki memori lokal, tetapi dapat berkomunikasi dengan proses lain, baik pada CPU yang sama maupun melalui jaringan.
 - Setiap proses dapat memiliki banyak thread (paralelisme hibrida).
 - MPI adalah yang paling umum digunakan.
 - Klaster, superkomputer, dll.

### Scalability
Ukuran performa paralel:
- Skalabilitas Kuat (Strong Scaling): Waktu eksekusi berkurang secara berbanding terbalik dengan jumlah proses
- Masalah ukuran tetap.
- Skalabilitas Lemah (Weak Scaling): Waktu eksekusi tetap konstan seiring dengan peningkatan ukuran masalah dan jumlah proses secara proporsional.
- Masalah ukuran variabel.

### OpenMP

OpenMP didefinisikan oleh sekumpulan directives (perintah pengarah) yang dimasukkan ke dalam kode, yang saat proses kompilasi dapat diubah oleh compiler menjadi kode multi-threaded.

Contoh hello world sederhana dari OpenMP. Di sini kita mengambil jumlah thread dan mencetak ID unik yang diberikan ke masing-masing thread.
```fortran
program hello_world_omp
    
    use omp_lib

    implicit none
    integer :: num_threads, thread_id

    !$OMP parallel private(num_threads, thread_id)
    !$ num_threads = omp_get_num_threads()
    !$ thread_id = omp_get_thread_num()
    print *, 'Hello from thread number', thread_id + 1, &
             ' of ', num_threads, ' processes'

    !$OMP end parallel

end program hello_world_omp

```

```fortran
program yeval
   
   use omp_lib

   implicit none

   integer(kind=8), parameter :: n = 2**16
   integer(kind=4) :: i, nthreads
   real(kind=8), dimension(n) :: y
   real(kind=8) :: dx, x

   ! Specify number of threads to use:
   !$ print *, "How many threads to use? "
   !$ read *, nthreads
   !$ call omp_set_num_threads(nthreads)
   !$ print "('Using OpenMP with ',i3,' threads')", nthreads

   dx = 1.d0 / (n+1.d0)

   !$omp parallel do private(x) 
   do i=1, n
      x = i * dx
      y(i) = exp(x) * cos(x) * sin(x) * sqrt(5.d0 * x + 6.d0)
   enddo
   !$omp end parallel do

   print *, "Filled vector y of length", n

end program yeval
```
*Modified from amath 583 - R.J. LeVeque - http://faculty.washington.edu/rjl/classes/am583s2014/notes/openmp.html*

#### Fine-Grain vs. Coarse-Grain Parallelism

Pertimbangkan masalah normalisasi vektor yang memerlukan dua langkah:

1. Menghitung norm (panjang) dari vektor tersebut, dan

2. Membagi setiap entri dalam vektor dengan nilai norm tersebut.

Sayangnya, kita perlu melakukan perulangan pada setiap entri dalam vektor untuk menghitung norm sebelum kita dapat melakukan pembagian pada setiap entri. Ada dua cara untuk menangani masalah ini:

- Membiarkan compiler memutuskan thread mana yang mengambil entri mana (fine-grain) — sejumlah besar tugas-tugas kecil
- Membiarkan pemrogram mengontrol secara eksplisit entri mana yang ditangani oleh setiap thread (coarse-grain) — sejumlah kecil tugas-tugas besar.

```fortran
program fine_grain
   
    use omp_lib
    implicit none
    integer :: i, thread_num
    integer, parameter :: n = 1000
 
    real(kind=8), dimension(n) :: x, y
    real(kind=8) :: norm,ynorm
 
    integer :: nthreads
    
    ! Specify number of threads to use:
    nthreads = 1       ! need this value in serial mode
    !$ nthreads = 4    
    !$ call omp_set_num_threads(nthreads)
    !$ print "('Using OpenMP with ',i3,' threads')", nthreads

    ! Specify number of threads to use:
    !$ call omp_set_num_threads(4)
 
    ! initialize x:
    !$omp parallel do 
    do i=1,n
        x(i) = real(i, kind=8)  ! convert to double float
    enddo

    norm = 0.d0
    ynorm = 0.d0

    !$omp parallel private(i)

    !$omp do reduction(+ : norm)
    do i=1,n
        norm = norm + abs(x(i))
    enddo

     !$omp barrier   ! not needed (implicit)

    !$omp do reduction(+ : ynorm)
    do i=1,n
        y(i) = x(i) / norm
        ynorm = ynorm + abs(y(i))
    enddo
    
    !$omp end parallel

    print *, "norm of x = ",norm, "  n(n+1)/2 = ",n*(n+1)/2
    print *, 'ynorm should be 1.0:   ynorm = ', ynorm

end program fine_grain

```
*Modified from amath 583 - R.J. LeVeque - http://faculty.washington.edu/rjl/classes/am583s2014/notes/openmp.html*

```fortran
program fine_grain
   
    use omp_lib
    implicit none
    integer :: i, thread_num
    integer, parameter :: n = 1000
 
    real(kind=8), dimension(n) :: x, y
    real(kind=8) :: norm,ynorm
 
    integer :: nthreads
    
    ! Specify number of threads to use:
    nthreads = 1       ! need this value in serial mode
    !$ nthreads = 4    
    !$ call omp_set_num_threads(nthreads)
    !$ print "('Using OpenMP with ',i3,' threads')", nthreads

    ! Specify number of threads to use:
    !$ call omp_set_num_threads(4)
 
    ! initialize x:
    !$omp parallel do 
    do i=1,n
        x(i) = real(i, kind=8)  ! convert to double float
    enddo

    norm = 0.d0
    ynorm = 0.d0

    !$omp parallel private(i)

    !$omp do reduction(+ : norm)
    do i=1,n
        norm = norm + abs(x(i))
    enddo

     !$omp barrier   ! not needed (implicit)

    !$omp do reduction(+ : ynorm)
    do i=1,n
        y(i) = x(i) / norm
        ynorm = ynorm + abs(y(i))
    enddo
    
    !$omp end parallel

    print *, "norm of x = ",norm, "  n(n+1)/2 = ",n*(n+1)/2
    print *, 'ynorm should be 1.0:   ynorm = ', ynorm

end program fine_grain

```
*Modified from amath 583 - R.J. LeVeque - http://faculty.washington.edu/rjl/classes/am583s2014/notes/openmp.html*

### MPI
Standar Message Passing Interface (MPI) menentukan antarmuka pemrograman untuk berkomunikasi antarproses, yang dapat dilakukan di seluruh jaringan heterogen. Tidak seperti OpenMP yang menggunakan directives (perintah pengarah), MPI menggunakan serangkaian panggilan fungsi untuk berkomunikasi antarunit paralel. Hal ini menuntut koordinasi yang jauh lebih besar dari pemrogram dan cenderung mengarah pada paralelisme coarse-grain (butiran kasar).

#### Additional Considerations for MPI
- Topologi jaringan
- Komunikasi satu arah (one-sided) vs. dua arah (two-sided
- Operasi penghentian (halting/blocking) vs. komunikasi asinkron

#### Example:  Hello World (Fortran)

Contoh Hello World dari MPI. Setiap proses akan mencetak nomor prosesnya `proc_num` dan jumlah total proses `num_procs`.  Perhatikan bahwa kita perlu menjalankan ini menggunakan perintah `mpiexec -n 4 hello_world_mpi` agar program ini dapat bekerja.

Panggilan dasar MPI yang digunakan di sini:
 - `mpi_init(integer error)` - Harus dipanggil di awal semua program MPI.
 - `mpi_comm_size(MPI_Comm comm, integer num_processes, integer error)` - Memerlukan komunikator MPI `comm` an mengembalikan jumlah proses ke dalam variabel `num_processes`.
 - `mpi_comm_rank(MPI_Comm comm, integer process_id, integer error)` - Memerlukan komunikator MPI `comm` dan mengembalikan ID proses yang sedang berjalan ke dalam variabel
 - `mpi_finalize(integer error)` - Menghentikan MPI, harus dipanggil di akhir setiap program MPI.

Perhatikan bahwa semua definisi panggilan MPI dalam bahasa Fortran memiliki argumen `integer error` di bagian akhir, yang nilainya bergantung pada kesalahan apa yang mungkin terjadi (spesifik untuk setiap panggilan fungsi).

```fortran
program hello_world_mpi
    
    use mpi

    implicit none
    integer :: error, num_procs, proc_num

    call mpi_init(error)
    call mpi_comm_size(MPI_COMM_WORLD, num_procs, error)
    call mpi_comm_rank(MPI_COMM_WORLD, proc_num, error)

    print *, 'Hello from Process number', proc_num + 1, &
             ' of ', num_procs, ' processes'

    call mpi_finalize(error)

end program hello_world_mpi
```

#### Example:  Hello World (Python)

Kita dapat menggunakan MPI dalam Python melalui paket `mpi4py` yang memiliki korespondensi hampir 1-ke-1 dengan antarmuka C.

Perlu diingat bahwa kode ini tidak dapat dijalankan langsung di dalam notebook. Kita harus menjalankannya dari command line (baris perintah) dengan perintah `mpiexec -n 4 python hello_world.py`.
```python
#!/usr/bin/env python
# encoding: utf-8

"""MPI Hello World from Python"""

from __future__ import absolute_import
from __future__ import print_function

from mpi4py import MPI

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

print('Hello from Process number %s of %s processes.' % (rank + 1, size))

```

#### Example: Computing $\pi$ (Fortran)

Hitung nilai $\pi$ secara paralel dengan mendekati nilai integral
$$
    \int^1_0 \frac{1}{1 + x^2} dx = \frac{\pi}{4}
$$
menggunakan aturan titik tengah (midpoint rule). Dalam kasus ini, kita akan membagi interval tersebut ke dalam sejumlah titik yang ditentukan oleh pengguna, dan membiarkan setiap prosesor mengambil jumlah titik yang sama (kurang lebih satu titik). Proses ini memiliki langkah-langkah sebagai berikut:

1. Proses 0 menanyakan kepada pengguna berapa banyak titik yang ingin digunakan.
2. Jumlah $N$ ini kemudian disiarkan (broadcast) ke setiap proses lainnya.
3. Setiap proses kemudian menghitung titik-titik yang menjadi tanggung jawabnya dan menghitung bagian jumlahnya masing-masing.
4. Hasil penjumlahan dari setiap proses kemudian dikurangi/digabungkan (reduced/added) bersama-sama dan dikirim kembali ke proses 0.

The following MPI functions are used below:

 - `mpi_bcast( void *buffer, integer count, integer datatype, int root, MPI_Comm comm, integer error)` dimana `count` adalah jumlah entri dalam buffer `buffer`adalah data yang dikirim, `datatype` adalah tipe data, `root` proses yang menyiarkan data tersebut, dan  `comm` adalah komunikator MPI (`MPI_COMM_WORLD`).
 - `mpi_Reduce(const void *send_buffer, void *receive_buffer, integer count, integer datatype, MPI_Op operation, integer root, MPI_Comm comm, integer error)` dimana `send_buffer` adalah penampung hasil akhir (pada proses yang dikirim), `receive_buffer` adalah penampung hasil akhir (pada proses yang dipilih) `count` dalah ukuran buffer, `datatype` adalah tipe data, `operation` adalah jenis operasi yang dilakukan (seperti penjumlahan), `root` adalah proses yang akan menerima hasil akhir, dan ``comm` adalah komunikator MPI.

```fortran
! Example program that computes pi in parallel
program compute_pi

    use mpi
    
    implicit none

    integer :: error, num_procs, proc_id, points_per_proc, n, i, start, end
    real(kind=8) :: x, dx, pi_sum, pi_sum_proc

    real(kind=8), parameter :: pi = 3.1415926535897932384626433832795d0

    call mpi_init(error)
    call mpi_comm_size(MPI_COMM_WORLD, num_procs, error)
    call mpi_comm_rank(MPI_COMM_WORLD, proc_id, error)

    ! Proc 0 will ask the user for the number of points
    if (proc_id == 0) then
        print *, "Using ",num_procs," processors"
        print *, "Input n ... "
        read *, n
    end if

    ! Broadcast to all procs; everybody gets the value of n from proc 0
    call mpi_bcast(n, 1, MPI_INTEGER, 0, MPI_COMM_WORLD, error)

    dx = 1.d0 / n

    ! Determine how many points to handle with each proc
    points_per_proc = (n + num_procs - 1) / num_procs
    ! Only print out the number of points per proc by process 0
    if (proc_id == 0) then   
        print *, "points_per_proc = ", points_per_proc
    end if

    ! Determine start and end index for this proc's points
    start = proc_id * points_per_proc + 1
    end = min((proc_id + 1) * points_per_proc, n)

    ! Diagnostic: tell the user which points will be handled by which proc
    print '("Process ",i2," will take i = ",i8," through i = ",i8)', &
          proc_id, start, end

    pi_sum_proc = 0.d0
    do i=start,end
        x = (i - 0.5d0) * dx
        pi_sum_proc = pi_sum_proc + 1.d0 / (1.d0 + x**2)
    enddo

    call MPI_REDUCE(pi_sum_proc, pi_sum, 1, MPI_DOUBLE_PRECISION, MPI_SUM, 0, &
                        MPI_COMM_WORLD, error)

    if (proc_id == 0) then
        print *, "The approximation to pi is ", 4.d0 * dx * pi_sum
        print *, "Difference = ", abs(pi - 4.d0 * dx * pi_sum)
    endif

    call mpi_finalize(error)

end program compute_pi
```

#### Example: Computing $\pi$ (Python)
```python
#!/usr/bin/env python
# encoding: utf-8

"""Note passing from Python"""

from __future__ import absolute_import
from __future__ import print_function

from mpi4py import MPI
import numpy

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

# This does not seem to work well in Python
# if rank == 0:
#     print("Using %s processes" % size)
#     N = input("  Input N...")

N = 100
N = comm.bcast(N, root=0)

dx = 1.0 / N

points_per_proc = (N + size - 1) / size

if rank == 0:
    print("Points/process = %s" % points_per_proc)

start = rank * points_per_proc + 1
end = min((rank + 1) * points_per_proc, N)

print("Process %s will take i = %s through i = %s" % (rank, start, end))

pi_sum_local = 0.0
for i in range(start, end + 1):
    x = (i - 0.5) * dx
    pi_sum_local += 1.0 / (1.0 + x**2)
pi_sum_local = numpy.array(pi_sum_local, dtype='d')

pi_sum = numpy.zeros(1)
comm.Reduce([pi_sum_local, MPI.DOUBLE], [pi_sum, MPI.DOUBLE],
            op=MPI.SUM, root=0)

if rank == 0:
    pi = 4.0 * dx * pi_sum
    print("The approximation to pi is %s" % (pi))
    print("Difference = %s" % (numpy.abs(numpy.pi - pi)))

```

#### Example:  Passing Notes (Fortran)

Contoh ini menunjukkan bagaimana satu proses dapat berkomunikasi dengan satu proses lainnya hingga pesan tersebut diteruskan ke proses terakhir. Dalam kasus ini, kita hanya meneruskan sebuah angka presisi ganda tunggal (sesuatu yang kurang menarik). Pada kondisi ini, setiap proses yang belum menerima pesan akan menunggu sampai ia menerimanya. Hal ini disebut sebagai blocking (pemblokiran) dan merupakan perilaku yang tidak diinginkan.

The only new function in this case is
 - `MPI_Recv(void *buffer, integer count, integer datatype, integer source, integer tag, MPI_Comm comm, MPI_Status status, integer error)` - Di sini kita kembali memiliki data yang diterima dalam `buffer`, jumlah nilai count, tipe data `datatype`, proses asal yaitu `source`, sebuah tanda pengenal `tag`, dan komunikator MPI `comm`. Nilai status menunjukkan status dari proses transfer tersebut. Perlu dicatat bahwa setelah kita memanggil `mpi_recv`, proses tersebut akan terblokir (block) sampai ia menerima data (hal yang kurang baik).
```fortran
! Demonstration for message passing between MPI processes
program note_passing

    use mpi

    implicit none

    integer :: proc_id, num_procs, error, tag
    real(kind=8) :: important_note
    integer, dimension(MPI_STATUS_SIZE) :: status

    call mpi_init(error)
    call mpi_comm_size(MPI_COMM_WORLD, num_procs, error)
    call mpi_comm_rank(MPI_COMM_WORLD, proc_id, error)

    if (num_procs == 1) then
        print *, "Only one process, cannot do anything."
        call MPI_FINALIZE(error)
        stop
    endif

    ! Not really important in this case but important to sort through messages
    tag = 42

    ! If we are process 0 then set the value to be passed
    if (proc_id == 0) then
        important_note = 2.718281828d0
        print '("Process ",i3," note = ",e18.8)', proc_id, important_note

        call mpi_send(important_note, 1, MPI_DOUBLE_PRECISION, 1, tag, &
                      MPI_COMM_WORLD, error)

    ! If we are one of the processes in between pass it on to the next process
    else if (proc_id < num_procs - 1) then

        call MPI_RECV(important_note, 1, MPI_DOUBLE_PRECISION, proc_id-1, tag, &
                      MPI_COMM_WORLD, status, error)

        print '("Process ",i3," received note = ",e18.8)', proc_id, important_note

        call mpi_send(important_note, 1, MPI_DOUBLE_PRECISION, proc_id + 1, &
                      tag, MPI_COMM_WORLD, error)

        print '("Process ",i3,"    sent note = ",e18.8)', proc_id, important_note

    ! If we are the last process in the class to find out, well...
    else if (proc_id == num_procs - 1) then

        call mpi_recv(important_note, 1, MPI_DOUBLE_PRECISION, proc_id - 1, &
                      tag, MPI_COMM_WORLD, status, error)

        print '("Process ",i3," ends up with note = ",e18.8)', proc_id, important_note
      endif

    call MPI_FINALIZE(error)

end program note_passing
```

#### Example:  Passing Notes (Python)

```python
#!/usr/bin/env python
# encoding: utf-8

"""Note passing from Python"""

from __future__ import absolute_import
from __future__ import print_function

from mpi4py import MPI
import numpy

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

tag = 42
N = 10

if rank == 0:
    data = numpy.arange(N, dtype=numpy.float64)
    print("Process %s note = %s" % (rank, data))
    # Note here that MPI datatype discovery is automatic
    comm.Send(data, dest=rank + 1, tag=tag)

elif rank < size - 1:
    data = numpy.empty(N, dtype=numpy.float64)
    comm.Recv(data, source=rank - 1, tag=tag)

    print("Process %s note = %s" % (rank, data))

    comm.Send(data, dest=rank + 1, tag=tag)

elif rank == size - 1:
    data = numpy.empty(N, dtype=numpy.float64)
    comm.Recv(data, source=rank - 1, tag=tag)

    print("Process %s note = %s" % (rank, data))

else:
    raise Exception("Invalid rank.")

```

### Example - Poisson Equation Solution Using Jacobi

Terakhir, mari kita lihat paralelisasi penyelesaian persamaan Poisson menggunakan iterasi Jacobi. Cara kita memparalelkan Jacobi cukup mudah; kita membagi evaluasi baris di antara thread/process yang tersedia. Ada tiga cara berbeda untuk melakukan ini yang akan kita bahas:
1. Butiran halus (fine-grained), di mana kita menempatkan directive OpenMP di sekitar setiap perulangan individu dan membiarkan thread mengambil pekerjaan saat tersedia.
2. Paralelisme butiran kasar (coarse-grained) dan OpenMP, di mana kita secara eksplisit memutuskan thread mana yang mendapatkan pekerjaan apa.
3. Paralelisme butiran kasar (coarse-grained) yang hampir identik dengan cara di atas, tetapi menggunakan MPI.

#### OpenMP - Fine-Grained Parallelism
```fortran
! Solve Poisson equation
!  u_{xx} = f(x)    x \in [a, b]
! with 
!  u(a) = alpha, u(b) = beta
! using Jacobi iteration and a fine-grain parallelism approach using OpenMP.
program jacobi_omp1

    use omp_lib
    
    implicit none
    
    ! Problem specification and storage
    real(kind=8) :: a, b, alpha, beta
    real(kind=8), dimension(:), allocatable :: x, u, u_new, f
    real(kind=8) :: dx, tolerance, du_max

    integer(kind=4) :: n, num_threads
    integer(kind=8) :: i, num_iterations
    integer(kind=8), parameter :: MAX_ITERATIONS = 100000
    real(kind=8) :: time(2)

    ! Boundaries
    a = 0.d0
    b = 1.d0
    alpha = 0.d0
    beta = 3.d0

    ! Specify number of threads to use:
    num_threads = 4
    !$ call omp_set_num_threads(num_threads)
    !$ print "('Using OpenMP with ',i3,' threads')", num_threads

    N = 100

    ! Allocate storage for boundary points too
    allocate(x(0:N + 1), u(0:N + 1), u_new(0:N + 1), f(0:N + 1))

    call cpu_time(time(1))

    ! grid spacing:
    dx = (b - a) / (N + 1.d0)

    ! Set iniital guess and construct the grid
    ! Note that here we are breaking up the problem already to the threads
    !$omp parallel do
    do i=0, N + 1
        ! grid points:
        x(i) = i * dx
        ! source term:
        f(i) = exp(x(i))
        ! initial guess (satisfies boundary conditions and sets them)
        u(i) = alpha + x(i) * (beta - alpha)
    enddo

    ! Tolerance
    tolerance = 0.1d0 * dx**2

    ! Main Jacobi iteration loop
    ! Copy old values to new
    num_iterations = 0
    du_max = 1d99
    do while (du_max >= tolerance .and. num_iterations <= MAX_ITERATIONS)
        du_max = 0.d0
        !$omp parallel do reduction(max : du_max)
        do i=1, N
            u_new(i) = 0.5d0 * (u(i-1) + u(i+1) - dx**2 * f(i))
            ! print *, abs(u(i) - u_old(i))
            du_max = max(du_max, abs(u_new(i) - u(i)))
        end do

        if (mod(num_iterations, 1000) == 0) then
            print *, "du_max, iteration = ", du_max, num_iterations
        end if

        ! Copy old data into new
        !$omp parallel do 
        do i=1, N
            u(i) = u_new(i)
        end do
        num_iterations = num_iterations + 1
    end do

    call cpu_time(time(2))
    if (num_iterations > MAX_ITERATIONS) then
        print *, "Iteration failed!"
        stop
    end if
    print '("CPU time = ",f12.8, " seconds")', time(2) - time(1)
    print *, "Total number of iterations: ", num_iterations

    ! Write out solution for checking
    open(unit=20, file="jacobi_omp1.txt", status="unknown")
    do i=0,n+1
        write(20,'(2e20.10)') x(i), u(i)
    enddo
    close(20)

end program jacobi_omp1
```

#### OpenMP - Coarse Grained Parallelism

Dalam pemrograman paralel, penjadwalan tugas yang eksplisit (paralelisme coarse-grained) memberikan kontrol lebih besar kepada pemrogram, namu


```fortran
program jacobi_omp2

    use omp_lib
    
    implicit none
    
    ! Problem specification and storage
    real(kind=8) :: a, b, alpha, beta
    real(kind=8), dimension(:), allocatable :: x, u, u_new, f
    real(kind=8) :: dx, tolerance, du_max, du_max_thread

    integer(kind=4) :: n, num_threads, points_per_thread, thread_id, start, end
    integer(kind=8) :: i, num_iterations
    integer(kind=8), parameter :: MAX_ITERATIONS = 100000
    real(kind=8) :: time(2)

    ! Boundaries
    a = 0.d0
    b = 1.d0
    alpha = 0.d0
    beta = 3.d0

    ! Specify number of threads to use:
    num_threads = 2
    !$ call omp_set_num_threads(num_threads)
    !$ print "('Using OpenMP with ',i3,' threads')", num_threads

    N = 100

    ! Allocate storage for boundary points too
    allocate(x(0:N + 1), u(0:N + 1), u_new(0:N + 1), f(0:N + 1))

    call cpu_time(time(1))

    ! grid spacing:
    dx = (b - a) / (N + 1.d0)

    ! Tolerance
    tolerance = 0.1d0 * dx**2

    ! Determine how many points to handle with each thread.
    ! Note that dividing two integers and assigning to an integer will
    ! round down if the result is not an integer.  
    ! This, together with the min(...) in the definition of iend below,
    ! insures that all points will get distributed to some thread.
    points_per_thread = (n + num_threads - 1) / num_threads
    print *, "points_per_thread = ", points_per_thread

    ! Start of the parallel block... 
    ! ------------------------------

    ! This is the only time threads are forked in this program:
    !$omp parallel private(thread_id, num_iterations, start, end, i,   &
    !$OMP                  du_max_thread)

    ! Set thread is, default to 0 if in serial
    thread_id = 0
    !$ thread_id = omp_get_thread_num()

    ! Determine start and end index
    start = thread_id * points_per_thread + 1
    end = min((thread_id + 1) * points_per_thread, N)

    ! Output some thread information and indexing
    !$omp critical
    print '("Thread ",i2," will take i = ",i6," through i = ",i6)', &
          thread_id, start, end
    !$omp end critical

    ! Set iniital guess and construct the grid
    do i=start, end
        ! grid points:
        x(i) = i * dx
        ! source term:
        f(i) = exp(x(i))
        ! initial guess (satisfies boundary conditions and sets them)
        u(i) = alpha + x(i) * (beta - alpha)
    enddo
    ! Note that the above does not set the boundaries, do this in a single thread
    !$omp single
    x(0) = a
    x(N + 1) = b
    u(0) = alpha
    u(N + 1) = beta
    !$omp end single nowait

    ! Main Jacobi iteration loop
    do num_iterations=1, MAX_ITERATIONS
        
        ! Make one thread reset the global du_max
        !$omp single
        du_max = 0.d0
        !$omp end single

        ! Private to each thread
        du_max_thread = 0.d0
        do i=start, end
            u_new(i) = 0.5d0 * (u(i-1) + u(i+1) - dx**2 * f(i))
            du_max_thread = max(du_max_thread, abs(u_new(i) - u(i)))
        end do

        ! Compute global du_max
        !$omp critical
        du_max = max(du_max, du_max_thread)
        !$omp end critical

        ! Make sure all threads are done contributing to du_max
        !$omp barrier

        ! Have one thread print out the convergence info
        !$omp single
        if (mod(num_iterations, 1000) == 0) then
            print '("After ",i8," iterations, dumax = ",d16.6,/)', num_iterations, du_max
        end if
        !$omp end single nowait

        ! Copy new data into u
        do i=start, end
            u(i) = u_new(i)
        end do

        ! Check exit criteria
        if (du_max < tolerance) then
            exit
        end if

        ! Make sure all threads are caught up to this point before starting
        ! another iteration
        !$omp barrier
    end do

    !$omp end parallel

    call cpu_time(time(2))
    if (num_iterations > MAX_ITERATIONS) then
        print *, "Iteration failed!"
        stop
    end if
    print '("CPU time = ",f12.8, " seconds")', time(2) - time(1)
    print *, "Total number of iterations: ", num_iterations

    ! Write out solution for checking
    open(unit=20, file="jacobi_omp2.txt", status="unknown")
    do i=0,n+1
        write(20,'(2e20.10)') x(i), u(i)
    enddo
    close(20)

end program jacobi_omp2

```

#### MPI
```fortran
program jacobi1d_mpi
    use mpi

    implicit none

    integer, parameter :: maxiter = 100000, nprint = 5000
    real (kind=8), parameter :: alpha = 20.d0, beta = 60.d0

    integer :: i, iter, istart, iend, points_per_task, itask, n
    integer :: ierr, ntasks, me, req1, req2
    integer, dimension(MPI_STATUS_SIZE) :: mpistatus
    real (kind = 8), dimension(:), allocatable :: f, u, uold
    real (kind = 8) :: x, dumax_task, dumax_global, dx, tol

    ! Initialize MPI; get total number of tasks and ID of this task
    call mpi_init(ierr)
    call mpi_comm_size(MPI_COMM_WORLD, ntasks, ierr)
    call mpi_comm_rank(MPI_COMM_WORLD, me, ierr)

    ! Ask the user for the number of points
    if (me == 0) then
        print *, "Input n ... "
        read *, n
    end if
    ! Broadcast to all tasks; everybody gets the value of n from task 0
    call mpi_bcast(n, 1, MPI_INTEGER, 0, MPI_COMM_WORLD, ierr)

    dx = 1.d0/real(n+1, kind=8)
    tol = 0.1d0*dx**2

    ! Determine how many points to handle with each task
    points_per_task = (n + ntasks - 1)/ntasks
    if (me == 0) then   ! Only one task should print to avoid clutter
        print *, "points_per_task = ", points_per_task
    end if

    ! Determine start and end index for this task's points
    istart = me * points_per_task + 1
    iend = min((me + 1)*points_per_task, n)

    ! Diagnostic: tell the user which points will be handled by which task
    print '("Task ",i2," will take i = ",i6," through i = ",i6)', &
        me, istart, iend


    ! Initialize:
    ! -----------

    ! This makes the indices run from istart-1 to iend+1
    ! This is more or less cosmetic, but makes things easier to think about
    allocate(f(istart-1:iend+1), u(istart-1:iend+1), uold(istart-1:iend+1))

    ! Each task sets its own, independent array
    do i = istart, iend
        ! Each task is a single thread with all its variables private
        ! so re-using the scalar variable x from one loop iteration to
        ! the next does not produce a race condition.
        x = dx*real(i, kind=8)
        f(i) = 100.d0*exp(x)               ! Source term
        u(i) = alpha + x*(beta - alpha)    ! Initial guess
    end do
    
    ! Set boundary conditions if this task is keeping track of a boundary
    ! point
    if (me == 0)        u(istart-1) = alpha
    if (me == ntasks-1) u(iend+1)   = beta


    ! Jacobi iteratation:
    ! -------------------

    do iter = 1, maxiter
        uold = u

        ! Send endpoint values to tasks handling neighboring sections
        ! of the array.  Note that non-blocking sends are used; note
        ! also that this sends from uold, so the buffer we're sending
        ! from won't be modified while it's being sent.
        !
        ! tag=1 is used for messages sent to the left
        ! tag=2 is used for messages sent to the right

        if (me > 0) then
            ! Send left endpoint value to process to the "left"
            call mpi_isend(uold(istart), 1, MPI_DOUBLE_PRECISION, me - 1, &
                1, MPI_COMM_WORLD, req1, ierr)
        end if
        if (me < ntasks-1) then
            ! Send right endpoint value to process on the "right"
            call mpi_isend(uold(iend), 1, MPI_DOUBLE_PRECISION, me + 1, &
                2, MPI_COMM_WORLD, req2, ierr)
        end if

        ! Accept incoming endpoint values from other tasks.  Note that
        ! these are blocking receives, because we can't run the next step
        ! of the Jacobi iteration until we've received all the
        ! incoming data.

        if (me < ntasks-1) then
            ! Receive right endpoint value
            call mpi_recv(uold(iend+1), 1, MPI_DOUBLE_PRECISION, me + 1, &
                1, MPI_COMM_WORLD, mpistatus, ierr)
        end if
        if (me > 0) then
            ! Receive left endpoint value
            call mpi_recv(uold(istart-1), 1, MPI_DOUBLE_PRECISION, me - 1, &
                2, MPI_COMM_WORLD, mpistatus, ierr)
        end if

        dumax_task = 0.d0   ! Max seen by this task

        ! Apply Jacobi iteration on this task's section of the array
        do i = istart, iend
            u(i) = 0.5d0*(uold(i-1) + uold(i+1) + dx**2*f(i))
            dumax_task = max(dumax_task, abs(u(i) - uold(i)))
        end do

        ! Take global maximum of dumax values
        call mpi_allreduce(dumax_task, dumax_global, 1, MPI_DOUBLE_PRECISION, &
            MPI_MAX, MPI_COMM_WORLD, ierr)
        ! Note that this MPI_ALLREDUCE call acts as an implicit barrier,
        ! since no process can return from it until all processes
        ! have called it.  Because of this, after this call we know
        ! that all the send and receive operations initiated at the
        ! top of the loop have finished -- all the MPI_RECV calls have
        ! finished in order for each process to get here, and if the
        ! MPI_RECV calls have finished, the corresponding MPI_ISEND
        ! calls have also finished.  Thus we can safely modify uold
        ! again.

        ! Also periodically report progress to the user
        if (me == 0) then
            if (mod(iter, nprint)==0) then
                print '("After ",i8," iterations, dumax = ",d16.6,/)', &
                    iter, dumax_global
            end if
        end if

        ! All tasks now have dumax_global, and can check for convergence
        if (dumax_global < tol) exit
    end do

    print '("Task number ",i2," finished after ",i9," iterations, dumax = ",&
            e16.6)', me, iter, dumax_global


    ! Output result:
    ! --------------

    ! Note: this only works if all processes share a file system
    ! and can all open and write to the same file!

    ! Synchronize to keep the next part from being non-deterministic
    call mpi_barrier(MPI_COMM_WORLD, ierr)

    ! Have each task output to a file in sequence, using messages to
    ! coordinate

    if (me == 0) then    ! Task 0 goes first
        ! Open file for writing, replacing any previous version:
        open(unit=20, file="heatsoln.txt", status="replace")
        write(20,*) "          x                  u"
        write(20, '(2e20.10)') 0.d0, u(0)    ! Boundary value at left end

        do i = istart, iend
            write(20, '(2e20.10)') i*dx, u(i)
        end do

        close(unit=20)
        ! Closing the file should guarantee that all the ouput 
        ! will be written to disk.
        ! If the file isn't closed before the next process starts writing,
        ! output may be jumbled or missing.

        ! Send go-ahead message to next task
        ! Only the fact that the message was sent is important, not its contents
        ! so we send the special address MPI_BOTTOM and length 0.
        ! tag=4 is used for this message.

        if (ntasks > 1) then
            call mpi_send(MPI_BOTTOM, 0, MPI_INTEGER, 1, 4, &
                          MPI_COMM_WORLD, ierr)
            endif

    else
        ! Wait for go-ahead message from previous task
        call mpi_recv(MPI_BOTTOM, 0, MPI_INTEGER, me - 1, 4, &
                          MPI_COMM_WORLD, mpistatus, ierr)
        ! Open file for appending; do not destroy previous contents
        open(unit=20, file="heatsoln.txt", status="old", access="append")
        do i = istart, iend
            write(20, '(2e20.10)') i*dx, u(i)
        end do

        ! Boundary value at right end:
        if (me == ntasks - 1) write(20, '(2e20.10)') 1.d0, u(iend+1)    

        ! Flush all pending writes to disk
        close(unit=20)

        if (me < ntasks - 1) then
            ! Send go-ahead message to next task
            call mpi_send(MPI_BOTTOM, 0, MPI_INTEGER, me + 1, 4, &
                          MPI_COMM_WORLD, ierr)
        end if
    end if

    ! Notify the user when all tasks have finished writing
    if (me == ntasks - 1) print *, "Solution is in heatsoln.txt"

    ! Close out MPI
    call mpi_finalize(ierr)

end program jacobi1d_mpi
```

In [ ]:
# Plotting code for each of the methods above
import os

import numpy
import matplotlib.pyplot as plt

u_true = lambda x: (4.0 - numpy.exp(1.0)) * x - 1.0 + numpy.exp(x)
x_fine = numpy.linspace(0.0, 1.0, 100)

base_path = "./src"
file_names = ["jacobi_omp1.txt", "jacobi_omp2.txt", "jacobi_mpi.txt"]
titles = ["OpenMP - Fine-Grained", "OpenMP - Coarse-Grained", "MPI"]
fig = plt.figure()
fig.set_figwidth(fig.get_figwidth() * 3)
for (i, title) in enumerate(titles):
    path = os.path.join(base_path, file_names[i])
    data = numpy.loadtxt(path)
    x = data[:, 0]
    U = data[:, 1]
    
    axes = fig.add_subplot(1, 3, i + 1)
    axes.plot(x, U, 'ro', label='computed')
    axes.plot(x_fine, u_true(x_fine), 'k', label="exact")
    axes.set_title(title)
    axes.set_xlabel('x')
    axes.set_ylabel('u(x)')

plt.show()

## Software Packages

### Linear Algebra
- BLAS — Basic Linear Algebra Subroutines
    - Level 1: Operasi skalar dan vektor.
    - Level 2: Operasi matriks-vektor.
    - Level 3: Operasi matriks-matriks.
- LAPACK — Linear Algebra Package
- ScaLAPACK — Parallel LAPACK

### Solving ODEs
ODEPACK

### Solving PDEs